# 6. Validación Estadística Formal

## 6.1 Verificación de Supuestos


In [1]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal, mannwhitneyu, chi2_contingency, shapiro
from scipy.stats import normaltest  # D'Agostino-Pearson
import scikit_posthocs as sp
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv('../data/processed/customers_segmentados.csv')
dfp = pd.read_pickle('../data/processed/df_processed.pkl')
# Merge cluster info
dfp = dfp.merge(df[['fullVisitorId', 'cluster', 'cluster_name']], on='fullVisitorId')

num_cols = ['n_sessions', 'totals.hits', 'totals.pageviews', 'bounce_prop',
            'weekend_prop', 'hour']
print(f"Dataset: {len(dfp):,} visitantes, {dfp['cluster'].nunique()} clusters")


Dataset: 9,996 visitantes, 4 clusters


In [2]:
# Test de D'Agostino-Pearson por variable y cluster
print("=== TEST DE NORMALIDAD D'AGOSTINO-PEARSON ===\n")
for col in num_cols:
    results = []
    for cid in sorted(dfp['cluster'].unique()):
        data = dfp.loc[dfp['cluster']==cid, col].dropna()
        if len(data) > 20:
            stat, p = normaltest(data)
            results.append(f"C{cid}: p={p:.2e} {'✗' if p<0.05 else '✓'}")
    print(f"{col}: {', '.join(results)}")

print("\nConclusión: NINGUNA variable es normal en ningún cluster.")
print("Decisión: usar Kruskal-Wallis (no paramétrico) sobre ANOVA.")


=== TEST DE NORMALIDAD D'AGOSTINO-PEARSON ===

n_sessions: C0: p=0.00e+00 ✗, C1: p=0.00e+00 ✗, C2: p=8.29e-75 ✗, C3: p=1.28e-125 ✗
totals.hits: C0: p=4.83e-286 ✗, C1: p=5.13e-287 ✗, C2: p=1.04e-58 ✗, C3: p=5.23e-66 ✗
totals.pageviews: C0: p=4.45e-255 ✗, C1: p=2.75e-230 ✗, C2: p=2.53e-50 ✗, C3: p=2.31e-51 ✗
bounce_prop: C0: p=0.00e+00 ✗, C1: p=1.94e-118 ✗, C2: p=8.05e-168 ✗, C3: p=1.90e-28 ✗
weekend_prop: C0: p=0.00e+00 ✗, C1: p=0.00e+00 ✗, C2: p=2.53e-213 ✗, C3: p=4.36e-32 ✗
hour: C0: p=1.92e-104 ✗, C1: p=4.12e-147 ✗, C2: p=1.82e-228 ✗, C3: p=4.07e-122 ✗

Conclusión: NINGUNA variable es normal en ningún cluster.
Decisión: usar Kruskal-Wallis (no paramétrico) sobre ANOVA.


## 6.2 Test Omnibus + Effect Size

Con n ≈ 10,000, **todo p-value sale ≈ 0**. Por eso reportamos centrado en **η²** (effect size), no solo significancia.


In [3]:
# Kruskal-Wallis + η²
print("=== KRUSKAL-WALLIS + EFFECT SIZE η² ===\n")
kw_results = []
for col in num_cols:
    groups = [dfp.loc[dfp['cluster']==c, col].dropna().values for c in sorted(dfp['cluster'].unique())]
    H, p = kruskal(*groups)
    n = sum(len(g) for g in groups)
    k = len(groups)
    eta2 = (H - k + 1) / (n - k)
    mag = 'Grande' if eta2 > 0.14 else 'Mediano' if eta2 > 0.06 else 'Pequeño' if eta2 > 0.01 else 'Trivial'
    kw_results.append({'Variable': col, 'H': round(H, 1), 'p-value': f'{p:.2e}',
                       'η²': round(eta2, 4), 'Magnitud': mag})

display(pd.DataFrame(kw_results).set_index('Variable'))
print("\nHallazgo: 5 de 6 variables con effect size grande.")
print("Excepción: hour con η² bajo (los clusters no se diferencian mucho por hora).")


=== KRUSKAL-WALLIS + EFFECT SIZE η² ===



,H,p-value,η²,Magnitud
Variable,,,,
n_sessions,5613.9,0.00e+00,0.5615,Grande
totals.hits,5283.6,0.00e+00,0.5285,Grande
totals.pageviews,5389.8,0.00e+00,0.5391,Grande
bounce_prop,2570.3,0.00e+00,0.2569,Grande
weekend_prop,5027.4,0.00e+00,0.5028,Grande
hour,174.6,1.28e-37,0.0172,Pequeño



Hallazgo: 5 de 6 variables con effect size grande.
Excepción: hour con η² bajo (los clusters no se diferencian mucho por hora).


## 6.3 Post-hoc de Dunn con Bonferroni


In [4]:
# Post-hoc de Dunn para cada variable
print("=== POST-HOC DE DUNN (BONFERRONI) ===\n")
for col in num_cols[:4]:  # Las más relevantes
    dunn = sp.posthoc_dunn(dfp, val_col=col, group_col='cluster', p_adjust='bonferroni')
    print(f"--- {col} ---")
    display(dunn.round(4))
    # Pares no significativos
    for i in range(4):
        for j in range(i+1, 4):
            if dunn.iloc[i, j] > 0.05:
                print(f"  ⚠ C{i} vs C{j}: p={dunn.iloc[i,j]:.4f} → NO significativo")
    print()


=== POST-HOC DE DUNN (BONFERRONI) ===

--- n_sessions ---


,0,1,2,3
0,1.0000,0.0,0.8153,0.0
1,0.0000,1.0,0.0000,0.0
2,0.8153,0.0,1.0000,0.0
3,0.0000,0.0,0.0000,1.0


  ⚠ C0 vs C2: p=0.8153 → NO significativo

--- totals.hits ---


,0,1,2,3
0,1.0,0.0,1.0,0.0
1,0.0,1.0,0.0,0.0
2,1.0,0.0,1.0,0.0
3,0.0,0.0,0.0,1.0


  ⚠ C0 vs C2: p=1.0000 → NO significativo

--- totals.pageviews ---


,0,1,2,3
0,1.0,0.0,1.0,0.0
1,0.0,1.0,0.0,0.0
2,1.0,0.0,1.0,0.0
3,0.0,0.0,0.0,1.0


  ⚠ C0 vs C2: p=1.0000 → NO significativo

--- bounce_prop ---


,0,1,2,3
0,1.0000,0.0000,0.1316,0.0000
1,0.0000,1.0000,0.0000,0.0018
2,0.1316,0.0000,1.0000,0.0000
3,0.0000,0.0018,0.0000,1.0000


  ⚠ C0 vs C2: p=0.1316 → NO significativo



## 6.4 Tests para Variables Categóricas


In [5]:
# Chi-cuadrado + Cramer's V
cat_cols = ['channelGrouping', 'device.deviceCategory']
print("=== CHI-CUADRADO + CRAMER'S V ===\n")
for col in cat_cols:
    ct = pd.crosstab(dfp['cluster'], dfp[col])
    chi2, p, dof, expected = chi2_contingency(ct)
    n = ct.sum().sum()
    k = min(ct.shape) - 1
    v = np.sqrt(chi2 / (n * k)) if k > 0 else 0
    mag = 'Fuerte' if v > 0.5 else 'Moderada' if v > 0.3 else 'Débil'
    print(f"{col}:")
    print(f"  χ² = {chi2:.1f}, p = {p:.2e}")
    print(f"  Cramer's V = {v:.3f} ({mag})")
    print()

print("Hallazgo: deviceCategory es el separador dominante (V fuerte), no el canal.")


=== CHI-CUADRADO + CRAMER'S V ===

channelGrouping:
  χ² = 1084.8, p = 2.28e-216
  Cramer's V = 0.190 (Débil)

device.deviceCategory:
  χ² = 9841.6, p = 0.00e+00
  Cramer's V = 0.702 (Fuerte)

Hallazgo: deviceCategory es el separador dominante (V fuerte), no el canal.


## 6.5 MANOVA Multivariado


In [6]:
from statsmodels.multivariate.manova import MANOVA

formula = ' + '.join([f'Q("{c}")' for c in num_cols])
manova = MANOVA.from_formula(f'{formula} ~ cluster', data=dfp.assign(cluster=dfp['cluster'].astype(str)))
result = manova.mv_test()
print("=== MANOVA ===")
print(result.summary())
print("\nWilks' Lambda < 1 con p < 0.001 → diferencia multivariada global confirmada.")


=== MANOVA ===
                    Multivariate linear model
                                                                  
------------------------------------------------------------------
         Intercept        Value  Num DF   Den DF   F Value  Pr > F
------------------------------------------------------------------
            Wilks' lambda 0.1510 6.0000 9987.0000 9361.5496 0.0000
           Pillai's trace 0.8490 6.0000 9987.0000 9361.5496 0.0000
   Hotelling-Lawley trace 5.6242 6.0000 9987.0000 9361.5496 0.0000
      Roy's greatest root 5.6242 6.0000 9987.0000 9361.5496 0.0000
------------------------------------------------------------------
                                                                  
------------------------------------------------------------------
        cluster         Value   Num DF   Den DF    F Value  Pr > F
------------------------------------------------------------------
          Wilks' lambda 0.1586 18.0000 28247.9870 1440.1739 0.0000
 